# 03 発言の長さと、BERT に入れるための分け方を確かめる(下調べ)

**問い**: 日本語 BERT(`tohoku-nlp/bert-base-japanese-v3`)は、一度に 512 トークン(文を区切った単位)までしか読めない。会議録の発言は、これに収まるか。収まらない発言を、どう分けるか。

**方法**: 前処理した発言(`speeches.parquet`)から 3,000件を無作為に選び、トークン数を数える。長い発言を文の切れ目で分ける方法(`src/embed/chunk.py`)を試す。

**結論(先に)**
- 約 16% の発言が 512 トークンを超え、超えた部分は全トークンの約 14% → 先頭だけを使うと、情報が多く失われる。党によって発言の長さが違うので、失われ方も党によって違う
- → 長い発言は、**文の切れ目で 512 以内のかたまりに分け**、かたまりごとに BERT に入れて、長さで重みを付けて平均する(04 で実行)
- 分け方を試したところ、かたまりの長さは最大 510(前後に付ける2つの記号を足して 512)で、上限に収まった

**流れ**: 準備 → 1. 発言の長さ → 2. 文の切れ目で分ける方法を試す

※ 2026-10-25 に、今の前処理の結果で上から順に実行し直した(数字は、最初に調べたときと少し違う)。

## 準備

- `AutoTokenizer`: 文を BERT の単位(トークン)に区切る道具
- `split_sentences`: 文を「。」「?」などで分ける / `chunk_ids`: 文をまとめて、上限以内のかたまり(トークンの番号の並び)にする(どちらも `src/embed/chunk.py`)

(出力の `TqdmWarning` は、進み具合の表示についての注意で、結果には影響しない)

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from src.embed.chunk import chunk_ids, split_sentences

c:\Users\taiyo\anaconda3\envs\survey_on_party_systems\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 発言の長さ(トークン数)

見るもの: トークン数の分布、512 を超える発言の割合、超えた部分が全体に占める割合、1トークンあたりの文字数、党ごとの中央値

In [2]:
q = pd.read_parquet("../data/processed/speeches.parquet")
tok = AutoTokenizer.from_pretrained("tohoku-nlp/bert-base-japanese-v3")

smp = q.sample(3000, random_state=0).copy()
smp["n_tok"] = [len(tok(t)["input_ids"]) for t in smp["text"]]
smp["n_char"] = smp["text"].str.len()

print(smp["n_tok"].describe(percentiles=[.5, .75, .9, .95, .99]).round(0))
print("\n512トークンを超える発言の割合:", round((smp["n_tok"] > 512).mean(), 3))
print("512を超えた部分が、全トークンに占める割合:",
      round((smp["n_tok"] - 512).clip(lower=0).sum() / smp["n_tok"].sum(), 3))
print("1トークンあたりの文字数:", round(smp["n_char"].sum() / smp["n_tok"].sum(), 2))
print("\n党ごとの中央値:")
print(smp.groupby("party")["n_tok"].median().sort_values())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (554 > 512). Running this sequence through the model will result in indexing errors


count    3000.0
mean      304.0
std       275.0
min         4.0
50%       238.0
75%       404.0
90%       616.0
95%       785.0
99%      1189.0
max      3623.0
Name: n_tok, dtype: float64

512トークンを超える発言の割合: 0.155
512を超えた部分が、全トークンに占める割合: 0.138
1トークンあたりの文字数: 1.51

党ごとの中央値:
party
保守     173.0
社民     197.0
共産     197.5
立憲     200.5
その他    239.0
中道     239.0
自民     252.0
国民     257.5
公明     261.5
維新     270.5
参政     278.0
れいわ    279.0
みらい    313.5
Name: n_tok, dtype: float64


**結果**
- トークン数: 中央値 238・平均 304・上位1% 1,189・最大 3,623
- **512 を超える発言は 15.5%**。超えた部分は全トークンの **13.8%**
- 1トークンは約 1.51字
- 党ごとの中央値は、保守 173 から みらい 313.5 まで差がある → 先頭 512 だけを使うと、長い発言の多い党ほど多く切り捨てられる

(最初の警告「554 > 512」は、BERT の上限より長い発言を数えたときの注意。ここでは長さを数えているだけなので問題ない)

## 2. 文の切れ目で分ける方法を試す

① 短い例文を、わざと小さい上限(8トークン)で分けて、文の切れ目で分かれるかを見る。② 3,000件で、かたまりの数と長さを数える。

In [3]:
q = pd.read_parquet("../data/processed/speeches.parquet")
tok = AutoTokenizer.from_pretrained("tohoku-nlp/bert-base-japanese-v3")

ex = "防衛費について伺います。増額には反対です。大臣の見解は?"
print(split_sentences(ex))
print([tok.decode(c) for c in chunk_ids(ex, tok, max_tokens=8)])

smp = q.sample(3000, random_state=0)
chunks = [chunk_ids(t, tok) for t in smp["text"]]
n_chunks = np.array([len(c) for c in chunks])
lens = np.array([len(x) for c in chunks for x in c])

print(
    "\nかたまりごとの数(発言ごと):", 
    pd.Series(n_chunks).describe(percentiles=[.5, .9, .99]).round(1).to_dict()
)
print(
    "1件あたり平均:", n_chunks.mean().round(2), 
    " / かたまりが0の発言:", (n_chunks == 0).sum()
)
print("かたまりの長さの最大:", lens.max(), "(510以下なら合格)")
print("全体のかたまり数の見積もり:", int(n_chunks.mean() * len(q)))


['防衛費について伺います。', '増額には反対です。', '大臣の見解は?']
['防衛 費 に つい て 伺い ます', '。', '増額 に は 反対 です 。', '大臣 の 見解 は ?']

かたまりごとの数(発言ごと): {'count': 3000.0, 'mean': 1.2, 'std': 0.5, 'min': 1.0, '50%': 1.0, '90%': 2.0, '99%': 3.0, 'max': 8.0}
1件あたり平均: 1.19  / かたまりが0の発言: 0
かたまりの長さの最大: 510 (510以下なら合格)
全体のかたまり数の見積もり: 104972


**結果**
- 例文は「。」の後で分かれた。1文目は上限に収まらないので、文の途中(「ます」と「。」の間)で切れた → **1文が上限より長いときだけ、文の途中で切る**
- 3,000件では、1件あたり平均 1.19かたまり(90% は2つ以下、最大 8)。かたまりが0の発言は無い
- かたまりの長さの最大は **510** → 前後に付ける記号(CLS・SEP)を足して 512 に収まる
- 全体では約 10.5万かたまりの見込み(04 では、そのときの 88,297件で 106,789かたまりだった)

## まとめ

- 発言の約 16% が BERT の上限を超えるので、文の切れ目で分けてから埋め込み、長さで重みを付けて平均する
- 分け方(`src/embed/chunk.py`)は、上限を守れていることを確かめた
- **限界**: かたまりに分けると、かたまりをまたぐ文脈は BERT から見えない